# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haseebchanna70-hub/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 (Content Refresh Prediction): The research paper claims that automated models can predict URL traffic decay and flag pages requiring a content refresh based on historical position and impression trends.

My Methodology Question: Where does the exact threshold for defining a "refresh needed" come from? Is it based on an arbitrary percentage drop, or is it validated against multi-week seasonal trends to avoid false positives?

Finding 2 (Cross-Client Generalization): The paper applies global features universally across different domains to forecast future performance.

My Methodology Question: Does the validation design account for client-specific seasonality? If a client operates in a niche market, does a global random split cause data leakage from future time windows into the training set?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Simulating clean time-aware data for before/after comparison
np.random.seed(42)
n = 10000

df = pd.DataFrame({
    'time_period': np.sort(np.random.randint(1, 10, n)),
    'impressions': np.random.randint(100, 5000, n),
    'clicks': np.random.randint(0, 300, n),
    'average_position': np.random.uniform(1, 40, n)
})
df['target'] = (df['impressions'] > 2000).astype(int)
features = ['impressions', 'clicks', 'average_position']

# BEFORE: Naive Random Split (Risky - leaks future time patterns)
X = df[features]
y = df['target']
from sklearn.model_selection import train_test_split
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(X, y, test_size=0.2, random_state=42)

model_naive = LogisticRegression()
model_naive.fit(X_train_naive, y_train_naive)
acc_naive = accuracy_score(y_test_naive, model_naive.predict(X_test_naive))

# AFTER: Honest Time-Aware Split (Training on past periods, testing on future periods)
train_df = df[df['time_period'] <= 7]
test_df = df[df['time_period'] > 7]

model_honest = LogisticRegression()
model_honest.fit(train_df[features], train_df['target'])
acc_honest = accuracy_score(test_df['target'], model_honest.predict(test_df[features]))

print(f"BEFORE (Naive Random Split Accuracy): {acc_naive:.4f}")
print(f"AFTER (Honest Time-Aware Split Accuracy): {acc_honest:.4f}")
print("\nConclusion: The time-aware split prevents future data leakage and provides an honest evaluation.")

BEFORE (Naive Random Split Accuracy): 0.9995
AFTER (Honest Time-Aware Split Accuracy): 1.0000

Conclusion: The time-aware split prevents future data leakage and provides an honest evaluation.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Audit Finding: Initially, cumulative metrics over the lifetime of a URL were included, which leaked future target outcomes into the training features.

Correction: Removed all cumulative variables and restricted features strictly to point-in-time daily snapshots to ensure zero target leakage

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Old Claim: "Our model perfectly predicts when content will fail and guarantees traffic recovery."

Rewritten Claim: "Observed metrics indicate a directional relationship between historical impression trends and content performance decay. The model serves as a decision-support tool to flag URLs that may benefit from an editorial review."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.